# HelmetGuard — Train your own motorcycle helmet detector

**CSST 106**. Run in Google Colab with **Runtime → Change runtime type → T4 GPU** (if available). Obtain a licensed YOLO detection dataset as ZIP, containing `data.yaml`, `images/`, and `labels/` (or Roboflow's `train/valid/test` format). This notebook trains on your dataset and evaluates on labeled images. **Do not claim a result before running the notebook.**


In [ ]:
!pip -q install ultralytics pyyaml
import torch
print('CUDA available:', torch.cuda.is_available())


## Upload your annotated dataset as a ZIP

Choose the exported **YOLOv8 object detection** dataset ZIP, not just unlabelled images. Keep the dataset's class-ID order.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
files_upload = files.upload()
zip_name = next((k for k in files_upload if k.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Upload a YOLO dataset ZIP containing data.yaml')
base = Path('/content/helmet_dataset'); base.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    # Reject paths trying to escape the project directory.
    for member in z.infolist():
        target = (base / member.filename).resolve()
        if not target.is_relative_to(base.resolve()):
            raise ValueError(f'Unsafe archive member: {member.filename}')
    z.extractall(base)
configs = sorted(base.rglob('data.yaml'))
if len(configs) != 1:
    raise RuntimeError(f'Expected exactly one data.yaml, found: {configs}')
config_path = configs[0]
print('Dataset file:', config_path)
print(config_path.read_text()[:1000])


## Normalize dataset paths and preserve label IDs
Roboflow exports sometimes use paths that point outside the extraction directory. This cell detects train/valid/test directories and creates `data_local.yaml` without modifying the original annotations.


In [ ]:
import yaml
original = yaml.safe_load(config_path.read_text())
root = config_path.parent
candidates = {'train':['train/images','images/train'], 'val':['valid/images','val/images','images/val'], 'test':['test/images','images/test']}
paths={}
for split, opts in candidates.items():
    found=next((rel for rel in opts if (root/rel).is_dir()),None)
    if not found: raise FileNotFoundError(f'Missing {split} images in {root}')
    paths[split]=found
config={'path':str(root), **paths, 'names':original['names'], 'nc':len(original['names'])}
config_path=root/'data_local.yaml'
config_path.write_text(yaml.safe_dump(config,sort_keys=False))
print('Classes:',config['names'])
print('Local dataset config:',config_path)
print('Paths:',paths)


## Fine-tune YOLO11 nano

These weights are trained with your own supplied labels. Training time depends on dataset size and GPU access.


In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
result = model.train(data=str(config_path), epochs=40, imgsz=640, batch=8, device=0 if torch.cuda.is_available() else 'cpu', project='/content/runs', name='helmetguard', plots=True, seed=42, patience=12)
best = Path(model.trainer.best)
assert best.exists(), 'Training did not save best.pt'
print('Best weights:', best)


## Evaluate on the held-out labeled test split
Only report test-set results after training. Save the confusion matrix and evaluation figures for your presentation.


In [ ]:
# Report ONLY results from the labeled test split (never substitute validation).
metrics=YOLO(str(best)).val(data=str(config_path), split='test', imgsz=640, plots=True, project='/content/reports', name='eval', exist_ok=True)
print('Evaluation split: test')
print('Precision:', float(metrics.box.mp))
print('Recall:', float(metrics.box.mr))
print('mAP50:', float(metrics.box.map50))
print('mAP50-95:', float(metrics.box.map))


## Download your trained checkpoint

Save `best.pt` as **`models/best.pt`** in the HelmetGuard project to prioritize your model over the starter model. Keep screenshots of the training charts and confusion matrix for your slides.


In [ ]:
from google.colab import files
files.download(str(best))
